In [0]:
# ===========================================
# Notebook Name:
# 03_validate_weekly_ml_pipeline
#
# Purpose:
# Final gate for the Weekly ML Pipeline
# (run_if: ALL_DONE). Checks that the
# deck_similarity pair count matches the
# expected N*(N-1)/2 combinatorics for the
# current deck_hash population, surfaces new
# archetype candidates (clusters whose
# representative deck did not exist in the
# previous run), and records this run's
# cluster_count / silhouette_score into
# pipeline_state so next week's
# 04_ml/01_validate_archetypes has a
# baseline to compare against.
#
# Input:
# pokemon.gold.deck_pokemon_features
# pokemon.gold.deck_similarity
# pokemon.gold.deck_archetypes
# pokemon.ops.pipeline_state
#
# Output:
# pokemon.ops.pipeline_state
# ===========================================

In [0]:
import json

from pyspark.sql import functions as F

DECK_POKEMON_FEATURES_TABLE = (
    "pokemon.gold.deck_pokemon_features"
)
DECK_SIMILARITY_TABLE = (
    "pokemon.gold.deck_similarity"
)
DECK_ARCHETYPES_TABLE = (
    "pokemon.gold.deck_archetypes"
)
PIPELINE_STATE_TABLE = (
    "pokemon.ops.pipeline_state"
)

PIPELINE_NAME = "weekly_ml_pipeline"
STAGE_NAME = "build_deck_archetypes"

print("Input :", DECK_POKEMON_FEATURES_TABLE)
print("Input :", DECK_SIMILARITY_TABLE)
print("Input :", DECK_ARCHETYPES_TABLE)
print("Output:", PIPELINE_STATE_TABLE)

In [0]:
deck_count = (
    spark.table(DECK_POKEMON_FEATURES_TABLE)
    .select("deck_hash")
    .distinct()
    .count()
)

expected_pair_count = (
    deck_count * (deck_count - 1) // 2
)

actual_pair_count = spark.table(
    DECK_SIMILARITY_TABLE
).count()

print("Deck count           :", deck_count)
print(
    "Expected similarity pairs:",
    expected_pair_count,
)
print(
    "Actual similarity pairs  :",
    actual_pair_count,
)

if actual_pair_count != expected_pair_count:
    raise ValueError(
        "deck_similarity pair count "
        f"({actual_pair_count}) does not match "
        f"the expected N*(N-1)/2 combinatorics "
        f"({expected_pair_count}) for "
        f"{deck_count} decks. Run "
        "03_build_deck_similarity."
    )

print(
    "Validation passed: deck_similarity pair "
    "count matches expected combinatorics"
)

In [0]:
archetype_summary_row = (
    spark.table(DECK_ARCHETYPES_TABLE)
    .select(
        "model_name",
        "cluster_count",
        "silhouette_score",
    )
    .distinct()
    .first()
)

if archetype_summary_row is None:
    raise ValueError(
        f"{DECK_ARCHETYPES_TABLE} is empty. "
        "Run 00_build_deck_archetypes first."
    )

current_model_name = (
    archetype_summary_row["model_name"]
)
current_cluster_count = (
    archetype_summary_row["cluster_count"]
)
current_silhouette_score = (
    archetype_summary_row["silhouette_score"]
)

print("Model name       :", current_model_name)
print("Cluster count    :", current_cluster_count)
print(
    "Silhouette score  :",
    current_silhouette_score,
)

In [0]:
current_representatives = {
    row["representative_deck_code"]
    for row in (
        spark.table(DECK_ARCHETYPES_TABLE)
        .select("representative_deck_code")
        .distinct()
        .collect()
    )
}

if spark.catalog.tableExists(
    PIPELINE_STATE_TABLE
):
    previous_state_row = (
        spark.table(PIPELINE_STATE_TABLE)
        .filter(
            (F.col("pipeline_name") == PIPELINE_NAME)
            & (F.col("stage_name") == STAGE_NAME)
        )
        .select("metrics_json")
        .first()
    )

else:
    previous_state_row = None

if (
    previous_state_row is None
    or previous_state_row["metrics_json"] is None
):
    print(
        "No previous run recorded for "
        f"{PIPELINE_NAME}/{STAGE_NAME}. "
        "Treating every current cluster as a "
        "new archetype candidate."
    )

    new_candidate_representatives = (
        current_representatives
    )

else:
    previous_metrics = json.loads(
        previous_state_row["metrics_json"]
    )

    previous_representatives = set(
        previous_metrics.get(
            "representative_deck_codes", []
        )
    )

    new_candidate_representatives = (
        current_representatives
        - previous_representatives
    )

    print(
        "Previous cluster_count   :",
        previous_metrics.get("cluster_count"),
    )
    print(
        "Previous silhouette_score:",
        previous_metrics.get(
            "silhouette_score"
        ),
    )

print(
    "New archetype candidates "
    "(representative deck_code not seen "
    "in the previous run):",
    len(new_candidate_representatives),
)

if new_candidate_representatives:
    display(
        spark.table(DECK_ARCHETYPES_TABLE)
        .filter(
            F.col(
                "representative_deck_code"
            ).isin(
                *new_candidate_representatives
            )
        )
    )

In [0]:
metrics = {
    "deck_count": deck_count,
    "cluster_count": current_cluster_count,
    "silhouette_score": (
        current_silhouette_score
    ),
    "representative_deck_codes": sorted(
        current_representatives
    ),
}

state_df = spark.createDataFrame(
    [{
        "pipeline_name": PIPELINE_NAME,
        "stage_name": STAGE_NAME,
        "last_run_status": "success",
        "watermark_value": current_model_name,
        "metrics_json": json.dumps(metrics),
        "consecutive_failure_count": 0,
    }],
    schema="""
    pipeline_name STRING,
    stage_name STRING,
    last_run_status STRING,
    watermark_value STRING,
    metrics_json STRING,
    consecutive_failure_count INT
    """,
).withColumn(
    "last_run_at", F.current_timestamp()
).withColumn(
    "last_success_at", F.current_timestamp()
)

state_df.createOrReplaceTempView(
    "staged_ml_pipeline_state"
)

spark.sql(f"""
MERGE INTO {PIPELINE_STATE_TABLE} AS target
USING staged_ml_pipeline_state AS source
ON  target.pipeline_name = source.pipeline_name
AND target.stage_name = source.stage_name

WHEN MATCHED THEN UPDATE SET
    target.last_run_at = source.last_run_at,
    target.last_run_status = source.last_run_status,
    target.last_success_at = source.last_success_at,
    target.watermark_value = source.watermark_value,
    target.metrics_json = source.metrics_json,
    target.consecutive_failure_count =
        source.consecutive_failure_count

WHEN NOT MATCHED THEN INSERT (
    pipeline_name,
    stage_name,
    last_run_at,
    last_run_status,
    last_success_at,
    watermark_value,
    metrics_json,
    consecutive_failure_count
)
VALUES (
    source.pipeline_name,
    source.stage_name,
    source.last_run_at,
    source.last_run_status,
    source.last_success_at,
    source.watermark_value,
    source.metrics_json,
    source.consecutive_failure_count
)
""")

print(
    "pipeline_state updated for "
    f"{PIPELINE_NAME}/{STAGE_NAME}: "
    f"cluster_count={current_cluster_count}, "
    "silhouette_score="
    f"{current_silhouette_score}"
)